<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 4 · Unsupervised Learning and Real Projects · Lesson 19 of 24

# k-Means Clustering: Finding Customer Groups

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Explain how k-means groups data without labels, choose the number of clusters with the elbow method and silhouette score, and describe each cluster in business terms.

## Learning without answers

Until now every example had a label (rent, default). In **unsupervised learning** there is no answer column. The model looks for **structure** by itself. The most common task is **clustering**: putting similar examples into groups.

A mobile-money company has 500 customers and wants to design different offers. Nobody has labelled the customers — the company wants the data to suggest the groups.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
def group(n, tx, amount, airtime):
    return np.column_stack([rng.normal(*tx, n), rng.normal(*amount, n), rng.normal(*airtime, n)])

data = np.vstack([group(250, (8, 3), (4000, 1500), (0.6, 0.1)),
                  group(150, (45, 10), (15000, 4000), (0.15, 0.08)),
                  group(100, (20, 6), (60000, 15000), (0.05, 0.04))])
customers = pd.DataFrame(data, columns=["tx_per_month", "avg_amount_rwf", "airtime_share"])
customers["tx_per_month"] = customers["tx_per_month"].clip(1).round()
customers["avg_amount_rwf"] = customers["avg_amount_rwf"].clip(500).round(-2)
customers["airtime_share"] = customers["airtime_share"].clip(0, 1).round(2)
customers = customers.sample(frac=1, random_state=1).reset_index(drop=True)   # shuffle
print(customers.shape)
customers.head()

The three features are transactions per month, average transaction amount and the share of transactions spent on airtime.

## How k-means works

1. Choose **k**, the number of clusters.
2. Place k **centroids** (centre points) at random.
3. **Assign** every customer to the nearest centroid.
4. **Move** each centroid to the average of its customers.
5. Repeat steps 3–4 until nothing changes.

Because the start is random, scikit-learn runs the whole thing several times (`n_init=10`) and keeps the best result. k-means uses **distances**, so scale first.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X_scaled = StandardScaler().fit_transform(customers)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
customers["segment"] = kmeans.fit_predict(X_scaled)
print(customers["segment"].value_counts())

## Describe the clusters

A cluster number means nothing until you describe it:

In [ ]:
profile = customers.groupby("segment").mean().round(1)
print(profile)

You should find three clear groups:

| Segment (your numbers may differ) | Behaviour | A name for the business |
|---|---|---|
| About 250 customers | ~8 transactions a month, ~4,000 RWF each, mostly airtime | **Occasional airtime users** |
| About 150 customers | ~44 transactions, ~15,000 RWF each | **Busy everyday users** (small traders) |
| About 100 customers | ~20 transactions, ~60,000 RWF each, little airtime | **Big-ticket payers** (businesses, school fees) |

Cluster numbers (0, 1, 2) are arbitrary — they can change between runs. The **descriptions** are what matter. Naming the clusters is a human job.

## How many clusters? The elbow method

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

ks = range(2, 8)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

plt.plot(ks, inertias, marker="o")
plt.xlabel("k")
plt.ylabel("inertia (within-cluster spread)")
plt.title("Elbow method")
plt.show()
for k, s in zip(ks, silhouettes):
    print(k, round(s, 3))

- **Inertia** is the total squared distance from each point to its centroid. It always falls as k grows, so look for the **elbow** — where the drop suddenly slows. Here the elbow is clearly at **k = 3**.
- The **silhouette score** (−1 to 1) measures how well each point fits its own cluster compared with the next one. Higher is better. It peaks at **k = 3** (about **0.74**).

Always combine the numbers with **business sense**: 3 offers may be practical, 12 may not.

## See the clusters

In [ ]:
plt.scatter(customers["tx_per_month"], customers["avg_amount_rwf"],
            c=customers["segment"], s=12)
plt.xlabel("Transactions per month")
plt.ylabel("Average amount (RWF)")
plt.title("Customer segments")
plt.show()

## Limits of k-means

- You must choose k.
- It finds **round**, similar-sized groups; it struggles with long, curved or very unequal clusters.
- Outliers pull centroids away.
- **DBSCAN** (density-based) finds clusters of any shape and marks outliers; **hierarchical clustering** builds a tree of clusters. Both are in `sklearn.cluster`.

## Other unsupervised uses

- **Anomaly detection** — `IsolationForest` flags unusual transactions (possible fraud) without fraud labels.
- **Dimensionality reduction** — PCA, next lesson.

## Key terms

- **Unsupervised learning** — finding structure in data without labels.
- **Clustering** — grouping similar examples.
- **Centroid** — the centre of a cluster.
- **Inertia** — within-cluster squared distances.
- **Elbow method / silhouette score** — ways to choose k.

**Takeaway:** k-means groups similar customers around centroids; choose k with the elbow, the silhouette score and business sense, then describe each group in plain words.

## Try it yourself

1. Run k-means with k = 4. Which segment splits in two? Is the split useful for the business?
2. Run k-means on the **unscaled** data. Compare the segment sizes with the scaled result and explain the difference.
3. Use `IsolationForest(random_state=42)` on the scaled data and look at the 10 most unusual customers.
4. Write one marketing idea for each of the three segments.

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*